# Initial Cleaning of the Merged Master Dataset

This notebook performs the first stage of cleaning and inspection for the merged thesis dataset.

It is designed to:
- load the merged dataset
- inspect column names, data types, and general data quality
- review columns in a structured way
- apply only basic and safe cleaning steps
- avoid model-specific preprocessing at this stage

The goal is to make the dataset easier to understand before any later feature engineering, imputation strategy, encoding, or modeling decisions.

In [81]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

## Load Dataset

The merged dataset combines price observations with Traficom-based features. A copy of the raw data is kept so that all cleaning steps remain explicit and easy to review.

In [82]:
data_path = Path("/Users/riteshbhandari/Documents/Dokumentit – Ritesh - MacBook Pro/GitHub/DPPM/datasets/merged/price_traficom_merged.csv")

if not data_path.exists():
    raise FileNotFoundError(f"Dataset not found: {data_path}")

df_raw = pd.read_csv(data_path, skipinitialspace=True)
df = df_raw.copy()

print(f"Dataset path: {data_path}")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")

display(df.head())

Dataset path: /Users/riteshbhandari/Documents/Dokumentit – Ritesh - MacBook Pro/GitHub/DPPM/datasets/merged/price_traficom_merged.csv
Rows: 11,374
Columns: 56


,product_id,part_name,price,quality_grade,oem_number,mileage,brand,model,category,subcategory,scrape_date,year_start,year_end,year_span,year_mid,brand_merge_key,model_merge_key,repair_status,brand_is_known_model_family,model_looks_like_part_taxonomy,model_family_clean,model_total_registered,model_median_vehicle_age,model_mean_vehicle_age,model_median_mileage,model_mean_mileage,model_median_engine_cc,model_median_power_kw,model_median_mass_kg,model_share_of_market,model_share_within_brand,model_share_over_10y,model_share_over_200k_km,model_automatic_share,model_petrol_share,model_diesel_share,model_ev_share,model_hybrid_share,model_turbo_share,brand_total_registered,brand_median_vehicle_age,brand_mean_vehicle_age,brand_median_mileage,brand_mean_mileage,brand_median_engine_cc,brand_median_power_kw,brand_median_mass_kg,brand_share_of_market,brand_share_over_10y,brand_share_over_200k_km,brand_automatic_share,brand_petrol_share,brand_diesel_share,brand_ev_share,brand_hybrid_share,brand_turbo_share
0,"63,136,980.000","Contact roll Airbag - , e-",177.600,A2,FI02042722A,"224,000.000",vw,golf,airbag,contact roll airbag,2026-02-03,2013,2020,7,"2,016.500",volkswagen,golf,original_valid,True,False,golf,86762,13.000,13.749,"186,801.000","195,467.968","1,395.000",81.000,"1,330.000",0.034,0.311,0.652,0.433,0.383,0.786,0.170,0.013,0.068,0.726,279311,11.000,12.154,"177,745.000","192,564.366","1,498.000",85.000,"1,400.000",0.109,0.538,0.387,0.478,0.657,0.246,0.075,0.073,0.686
1,"64,390,201.000","Contact roll Airbag - , e-",177.600,A2,FI05351686A,"272,000.000",vw,golf,airbag,contact roll airbag,2026-02-03,2013,2020,7,"2,016.500",volkswagen,golf,original_valid,True,False,golf,86762,13.000,13.749,"186,801.000","195,467.968","1,395.000",81.000,"1,330.000",0.034,0.311,0.652,0.433,0.383,0.786,0.170,0.013,0.068,0.726,279311,11.000,12.154,"177,745.000","192,564.366","1,498.000",85.000,"1,400.000",0.109,0.538,0.387,0.478,0.657,0.246,0.075,0.073,0.686
2,"58,952,159.000","Contact roll Airbag - , e-",177.600,A2,FI27837687A,"134,000.000",vw,golf,airbag,contact roll airbag,2026-02-03,2013,2020,7,"2,016.500",volkswagen,golf,original_valid,True,False,golf,86762,13.000,13.749,"186,801.000","195,467.968","1,395.000",81.000,"1,330.000",0.034,0.311,0.652,0.433,0.383,0.786,0.170,0.013,0.068,0.726,279311,11.000,12.154,"177,745.000","192,564.366","1,498.000",85.000,"1,400.000",0.109,0.538,0.387,0.478,0.657,0.246,0.075,0.073,0.686
3,"63,225,719.000","Contact roll Airbag - , e-",177.600,A2,FI27837687A,"253,000.000",vw,golf,airbag,contact roll airbag,2026-02-03,2013,2020,7,"2,016.500",volkswagen,golf,original_valid,True,False,golf,86762,13.000,13.749,"186,801.000","195,467.968","1,395.000",81.000,"1,330.000",0.034,0.311,0.652,0.433,0.383,0.786,0.170,0.013,0.068,0.726,279311,11.000,12.154,"177,745.000","192,564.366","1,498.000",85.000,"1,400.000",0.109,0.538,0.387,0.478,0.657,0.246,0.075,0.073,0.686
4,"64,336,640.000","Contact roll Airbag - , e-",177.600,A2,FI09389104A,"152,000.000",vw,golf,airbag,contact roll airbag,2026-02-03,2013,2020,7,"2,016.500",volkswagen,golf,original_valid,True,False,golf,86762,13.000,13.749,"186,801.000","195,467.968","1,395.000",81.000,"1,330.000",0.034,0.311,0.652,0.433,0.383,0.786,0.170,0.013,0.068,0.726,279311,11.000,12.154,"177,745.000","192,564.366","1,498.000",85.000,"1,400.000",0.109,0.538,0.387,0.478,0.657,0.246,0.075,0.073,0.686


## Initial Overview

This first overview checks dataset size, column names, and the starting data types before any cleaning changes are applied.

In [83]:
print("Original column names:")
print(df.columns.tolist())
print()

df.info()

Original column names:
['product_id', 'part_name                                         ', 'price ', 'quality_grade', 'oem_number ', 'mileage ', 'brand ', 'model  ', 'category                                 ', 'subcategory                            ', 'scrape_date', 'year_start', 'year_end', 'year_span', 'year_mid', 'brand_merge_key', 'model_merge_key', 'repair_status ', 'brand_is_known_model_family', 'model_looks_like_part_taxonomy', 'model_family_clean', 'model_total_registered', 'model_median_vehicle_age', 'model_mean_vehicle_age', 'model_median_mileage', 'model_mean_mileage', 'model_median_engine_cc', 'model_median_power_kw', 'model_median_mass_kg', 'model_share_of_market', 'model_share_within_brand', 'model_share_over_10y', 'model_share_over_200k_km', 'model_automatic_share', 'model_petrol_share', 'model_diesel_share', 'model_ev_share    ', 'model_hybrid_share', 'model_turbo_share ', 'brand_total_registered', 'brand_median_vehicle_age', 'brand_mean_vehicle_age', 'brand_median_m

## Duplicate Observation Warning

Some observations that may look like duplicates can represent the **same listing or part observed repeatedly over time**. In this project, these repeated observations should be preserved unless there is a very clear accidental exact-duplicate issue.

For that reason, this notebook only **reports** duplicate patterns. It does **not** automatically drop duplicate rows.

## Basic Cleaning

The cleaning below is intentionally conservative. It standardizes names and obvious formatting issues without making modeling decisions or deleting information.

In [84]:
original_columns = df.columns.tolist()

# Standardize column names for easier handling.
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", "_", regex=True)
)

object_columns = df.select_dtypes(include=["object"]).columns.tolist()
for column in object_columns:
    df[column] = df[column].astype("string").str.strip()

# Replace empty strings with missing values after whitespace stripping.
df[object_columns] = df[object_columns].replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})

numeric_name_fragments = [
    "price",
    "mileage",
    "year",
    "age",
    "registered",
    "engine_cc",
    "power_kw",
    "mass_kg",
    "share",
    "span",
    "mid",
]

candidate_numeric_columns = [
    column
    for column in df.columns
    if any(fragment in column for fragment in numeric_name_fragments)
    and column not in {"product_id"}
]

for column in candidate_numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

date_like_columns = [column for column in df.columns if "date" in column]
for column in date_like_columns:
    df[column] = pd.to_datetime(df[column], errors="coerce")

if "product_id" in df.columns:
    product_id_numeric = pd.to_numeric(df["product_id"], errors="coerce")
    df["product_id"] = product_id_numeric.astype("Int64").astype("string")

print("Column names before cleaning:")
print(original_columns)
print()
print("Column names after cleaning:")
print(df.columns.tolist())
print()
print(f"Object/string columns cleaned: {len(object_columns)}")
print(f"Numeric columns coerced where obvious: {len(candidate_numeric_columns)}")
print(f"Date-like columns parsed where obvious: {len(date_like_columns)}")

Column names before cleaning:
['product_id', 'part_name                                         ', 'price ', 'quality_grade', 'oem_number ', 'mileage ', 'brand ', 'model  ', 'category                                 ', 'subcategory                            ', 'scrape_date', 'year_start', 'year_end', 'year_span', 'year_mid', 'brand_merge_key', 'model_merge_key', 'repair_status ', 'brand_is_known_model_family', 'model_looks_like_part_taxonomy', 'model_family_clean', 'model_total_registered', 'model_median_vehicle_age', 'model_mean_vehicle_age', 'model_median_mileage', 'model_mean_mileage', 'model_median_engine_cc', 'model_median_power_kw', 'model_median_mass_kg', 'model_share_of_market', 'model_share_within_brand', 'model_share_over_10y', 'model_share_over_200k_km', 'model_automatic_share', 'model_petrol_share', 'model_diesel_share', 'model_ev_share    ', 'model_hybrid_share', 'model_turbo_share ', 'brand_total_registered', 'brand_median_vehicle_age', 'brand_mean_vehicle_age', 'brand_m

## Column Summary

A compact summary table helps identify the role and quality of each column. The conceptual column type is partly heuristic, so it should be treated as a review aid rather than a final classification.

In [85]:
def conceptual_column_type(series: pd.Series) -> str:
    column_name = series.name.lower()

    if column_name == "price":
        return "target"
    if column_name.endswith("_id") or column_name == "product_id":
        return "id"
    if pd.api.types.is_datetime64_any_dtype(series):
        return "date"
    if pd.api.types.is_numeric_dtype(series):
        return "numeric"
    if any(token in column_name for token in ["date", "time"]):
        return "date"
    if any(token in column_name for token in ["name", "description", "comment", "text", "number"]):
        return "text"
    if series.nunique(dropna=True) <= 30:
        return "categorical"
    return "text"


def sample_values(series: pd.Series, max_values: int = 5) -> str:
    values = series.dropna().astype(str).unique().tolist()[:max_values]
    return ", ".join(values)


column_summary = pd.DataFrame(
    {
        "column_name": df.columns,
        "dtype": [str(df[column].dtype) for column in df.columns],
        "conceptual_type": [conceptual_column_type(df[column]) for column in df.columns],
        "missing_count": [int(df[column].isna().sum()) for column in df.columns],
        "missing_percentage": [df[column].isna().mean() * 100 for column in df.columns],
        "unique_values": [int(df[column].nunique(dropna=True)) for column in df.columns],
        "sample_values": [sample_values(df[column]) for column in df.columns],
    }
).sort_values(["conceptual_type", "column_name"]).reset_index(drop=True)


display(column_summary)

,column_name,dtype,conceptual_type,missing_count,missing_percentage,unique_values,sample_values
0,brand,string,categorical,0,0.000,3,"vw, toyota, skoda"
1,brand_is_known_model_family,string,categorical,0,0.000,2,"True, False"
2,brand_merge_key,string,categorical,0,0.000,3,"volkswagen, toyota, skoda"
3,category,string,categorical,0,0.000,7,"airbag, brakes, electric / transmitter / databox / sensor, engine, fuel"
4,model,string,categorical,0,0.000,3,"golf, corolla, octavia"
5,model_family_clean,string,categorical,0,0.000,3,"golf, corolla, octavia"
6,model_looks_like_part_taxonomy,string,categorical,0,0.000,1,False
7,model_merge_key,string,categorical,0,0.000,3,"golf, corolla, octavia"
8,quality_grade,string,categorical,0,0.000,10,"A2, A1, B2, A3, B1"
9,repair_status,string,categorical,0,0.000,1,original_valid


## Column-by-Column Inspection

The next cells inspect the dataset by column type. This is useful for spotting unusual ranges, unexpected values, and columns that may need more careful treatment later.

In [86]:
numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()

numeric_summary = pd.DataFrame(
    {
        "column_name": numeric_columns,
        "missing_count": [int(df[column].isna().sum()) for column in numeric_columns],
        "missing_percentage": [df[column].isna().mean() * 100 for column in numeric_columns],
        "min": [df[column].min() for column in numeric_columns],
        "max": [df[column].max() for column in numeric_columns],
        "mean": [df[column].mean() for column in numeric_columns],
        "median": [df[column].median() for column in numeric_columns],
    }
).sort_values("column_name")

print(f"Numeric columns: {len(numeric_columns)}")
display(numeric_summary)

Numeric columns: 41


,column_name,missing_count,missing_percentage,min,max,mean,median
35,brand_automatic_share,0,0.000,0.024,0.592,0.357,0.478
37,brand_diesel_share,0,0.000,0.069,0.269,0.192,0.246
38,brand_ev_share,0,0.000,0.005,0.075,0.048,0.067
39,brand_hybrid_share,0,0.000,0.073,0.323,0.163,0.082
28,brand_mean_mileage,0,0.000,"179,263.663","193,594.446","188,668.733","192,564.366"
26,brand_mean_vehicle_age,0,0.000,9.062,13.211,11.532,12.154
29,brand_median_engine_cc,0,0.000,"1,498.000","1,598.000","1,532.702","1,498.000"
31,brand_median_mass_kg,0,0.000,"1,345.000","1,400.000","1,372.920","1,375.000"
27,brand_median_mileage,0,0.000,"162,500.000","179,335.500","173,422.127","177,745.000"
30,brand_median_power_kw,0,0.000,81.000,96.000,87.129,85.000


In [87]:
categorical_columns = [
    column
    for column in df.columns
    if df[column].dtype in ["object", "string", "category", "boolean"]
]

print(f"Categorical/text-like columns selected for quick review: {len(categorical_columns)}")

for column in categorical_columns:
    print(f"\n--- {column} ---")
    print(f"Missing values: {df[column].isna().sum():,} ({df[column].isna().mean() * 100:.2f}%)")
    display(df[column].value_counts(dropna=False).head(10).rename("count").to_frame())


Categorical/text-like columns selected for quick review: 14

--- product_id ---
Missing values: 0 (0.00%)


,count
product_id,
54142195,7
64248645,7
53589483,7
54321498,7
60058883,7
64935321,7
66223598,7
64546407,7
59131986,7



--- part_name ---
Missing values: 0 (0.00%)


,count
part_name,
Suspension -(Rear),140
Shock absorbers rear -(Rear),133
Drive shaft -(Left front),132
Trailing link rear -(Left),131
Curtain airbags -(Right),117
Drive shaft -(Right front),113
Trailing link rear -(Right),113
Hub rear -(Rear),108
Wheel bearing spindle shaft -(Left rear),106



--- quality_grade ---
Missing values: 0 (0.00%)


...

,count
brand_is_known_model_family,
True,7737
False,3637



--- model_looks_like_part_taxonomy ---
Missing values: 0 (0.00%)


,count
model_looks_like_part_taxonomy,
False,11374



--- model_family_clean ---
Missing values: 0 (0.00%)


,count
model_family_clean,
corolla,3947
golf,3790
octavia,3637


In [88]:
date_columns = [column for column in df.columns if pd.api.types.is_datetime64_any_dtype(df[column])]

if date_columns:
    date_summary = pd.DataFrame(
        {
            "column_name": date_columns,
            "missing_count": [int(df[column].isna().sum()) for column in date_columns],
            "missing_percentage": [df[column].isna().mean() * 100 for column in date_columns],
            "min_date": [df[column].min() for column in date_columns],
            "max_date": [df[column].max() for column in date_columns],
        }
    ).sort_values("column_name")
    display(date_summary)
else:
    print("No date columns were parsed successfully in the basic cleaning step.")

,column_name,missing_count,missing_percentage,min_date,max_date
0,scrape_date,0,0.000,2026-02-03,2026-02-18


## Duplicate Checks

These checks are purely diagnostic. They help assess whether there are exact row duplicates or repeated identifiers, but no rows are removed automatically.

In [89]:
full_row_duplicates = int(df.duplicated().sum())
print(f"Fully duplicated rows: {full_row_duplicates:,}")

id_columns = [column for column in df.columns if column.endswith("_id") or column == "product_id"]

if id_columns:
    duplicate_id_summary = pd.DataFrame(
        {
            "column_name": id_columns,
            "duplicate_values": [int(df[column].duplicated(keep=False).sum()) for column in id_columns],
            "unique_values": [int(df[column].nunique(dropna=True)) for column in id_columns],
            "missing_count": [int(df[column].isna().sum()) for column in id_columns],
        }
    ).sort_values("column_name")
    display(duplicate_id_summary)
else:
    print("No ID-like columns were detected.")

Fully duplicated rows: 0


,column_name,duplicate_values,unique_values,missing_count
0,product_id,10540,2619,0


## Missing Values Review

Missing values are reviewed after the basic cleaning step so that empty strings and parsing failures are already reflected in the totals.

In [90]:
missing_values_review = (
    pd.DataFrame(
        {
            "column_name": df.columns,
            "missing_count": [int(df[column].isna().sum()) for column in df.columns],
            "missing_percentage": [df[column].isna().mean() * 100 for column in df.columns],
        }
    )
    .sort_values(["missing_percentage", "missing_count"], ascending=[False, False])
    .reset_index(drop=True)
)

print("Columns with the highest missingness:")
display(missing_values_review.head(20))

Columns with the highest missingness:


,column_name,missing_count,missing_percentage
0,mileage,1115,9.803
1,product_id,0,0.000
2,part_name,0,0.000
3,price,0,0.000
4,quality_grade,0,0.000
5,oem_number,0,0.000
6,brand,0,0.000
7,model,0,0.000
8,category,0,0.000
9,subcategory,0,0.000


## Consistency Checks

The following quick checks are simple diagnostics for obvious structural issues in the cleaned dataset. They are kept separate from the main summary sections so the notebook remains easier to follow.

In [91]:
# Logical year consistency
((df["year_end"] < df["year_start"]).sum(), (df["year_span"] <= 0).sum())

(0, 0)

## Mileage Review

Mileage deserves a closer look because it is one of the most practically important variables in the dataset and may contain unusually small values that require cautious handling.

In [92]:
# Mileage tail inspection
df["mileage"].quantile([0, 0.01, 0.05, 0.5, 0.95, 0.99, 1.0])

0.000         1.000
0.010     2,000.000
0.050    16,000.000
0.500   120,000.000
0.950   324,000.000
0.990   433,000.000
1.000   919,294.000
Name: mileage, dtype: float64

In [93]:
# Correlation quick look for numeric columns
numeric_df = df.select_dtypes(include=["number"])
numeric_df.corr(numeric_only=True)["price"].sort_values(ascending=False)

price                       1.000
year_mid                    0.226
year_end                    0.226
year_start                  0.224
year_span                   0.099
model_share_within_brand    0.023
model_diesel_share          0.022
model_hybrid_share          0.022
model_automatic_share       0.022
model_turbo_share           0.022
model_median_mass_kg        0.022
brand_automatic_share       0.022
brand_turbo_share           0.022
brand_median_power_kw       0.021
brand_diesel_share          0.021
brand_ev_share              0.019
model_median_power_kw       0.019
model_share_of_market       0.019
model_total_registered      0.019
brand_median_mass_kg        0.014
model_ev_share              0.002
model_median_engine_cc     -0.002
brand_share_over_200k_km   -0.018
model_mean_mileage         -0.019
model_median_mileage       -0.020
brand_mean_mileage         -0.020
brand_hybrid_share         -0.020
brand_median_mileage       -0.020
brand_median_engine_cc     -0.020
model_share_ov

In [94]:
df.loc[df["mileage"] <= 1000, ["product_id", "part_name", "mileage", "brand", "model", "scrape_date"]]

,product_id,part_name,mileage,brand,model,scrape_date
122,65711229,"Adaptiv Farthållare Sensor - , e-","1,000.000",vw,golf,2026-02-03
154,54070103,"Other Control unit - , e-","1,000.000",vw,golf,2026-02-03
186,54410773,"Battery box / Fixing / Holder - , e-",1.000,vw,golf,2026-02-03
202,53919106,"Sensor ABS - , e-","1,000.000",vw,golf,2026-02-03
203,53919107,"Sensor ABS - , e-","1,000.000",vw,golf,2026-02-03
756,65711229,"Adaptiv Farthållare Sensor - , e-","1,000.000",vw,golf,2026-02-06
788,54070103,"Other Control unit - , e-","1,000.000",vw,golf,2026-02-06
820,54410773,"Battery box / Fixing / Holder - , e-",1.000,vw,golf,2026-02-06
836,53919106,"Sensor ABS - , e-","1,000.000",vw,golf,2026-02-06
837,53919107,"Sensor ABS - , e-","1,000.000",vw,golf,2026-02-06


In [95]:
low_mileage_count = (df["mileage"] <= 1000).sum()
print(low_mileage_count)

49


In [96]:
# Replace implausibly low mileage values (<= 1000) with missing values
df.loc[df["mileage"] <= 1000, "mileage"] = np.nan

# Check updated mileage summary
print(df["mileage"].describe())

count    10,210.000
mean    145,478.007
std     101,862.971
min       2,000.000
25%      68,000.000
50%     121,000.000
75%     204,000.000
max     919,294.000
Name: mileage, dtype: float64


In [97]:
# Inspect very low mileage rows
print(df.loc[df["mileage"] <= 5000, ["product_id", "part_name", "mileage", "brand", "model", "scrape_date"]].sort_values("mileage"))

      product_id                    part_name   mileage   brand    model  \
11333   54242387           Suspension -(Rear) 2,000.000   skoda  octavia   
9029    54224758                Brake servo - 2,000.000   skoda  octavia   
6249    53850683  Brake Caliper -(Right rear) 2,000.000  toyota  corolla   
6244    53860486   Brake Caliper -(Left rear) 2,000.000  toyota  corolla   
6185    53883759   Trailing link rear -(Left) 2,000.000  toyota  corolla   
...          ...                          ...       ...     ...      ...   
5668    53869465   Trailing link rear -(Left) 4,000.000  toyota  corolla   
5400    53848511            Engine Gasoline - 4,000.000  toyota  corolla   
5216    53867130   Brake Caliper -(Left rear) 4,000.000  toyota  corolla   
8148    53489119                   Tank lid - 4,000.000   skoda  octavia   
6245    53867130   Brake Caliper -(Left rear) 4,000.000  toyota  corolla   

      scrape_date  
11333  2026-02-18  
9029   2026-02-09  
6249   2026-02-12  
6244   

In [98]:
# Low-end mileage frequency table up to 20,000
print(df.loc[df["mileage"] <= 20000, "mileage"].value_counts().sort_index())

mileage
2,000.000     79
2,334.000     18
3,000.000      6
4,000.000     31
6,000.000     12
7,000.000     59
8,000.000     32
10,000.000    42
11,000.000    66
12,000.000    60
15,000.000    37
16,000.000    72
17,000.000    56
18,000.000    62
19,000.000    30
20,000.000    88
Name: count, dtype: int64


In [99]:
# Summary counts by threshold
for threshold in [1000, 2000, 3000, 5000, 7000, 10000, 15000, 20000]:
    count = (df["mileage"] <= threshold).sum()
    print(f"<= {threshold}: {count}")

<= 1000: 0
<= 2000: 79
<= 3000: 103
<= 5000: 134
<= 7000: 205
<= 10000: 279
<= 15000: 442
<= 20000: 750


In [100]:
# Inspect rows in the suspicious low range
print(
    df.loc[
        df["mileage"].between(1001, 10000),
        ["product_id", "part_name", "mileage", "brand", "model", "scrape_date"]
    ].sort_values("mileage")
)

     product_id                                part_name    mileage   brand  \
6244   53860486               Brake Caliper -(Left rear)  2,000.000  toyota   
6249   53850683              Brake Caliper -(Right rear)  2,000.000  toyota   
6185   53883759               Trailing link rear -(Left)  2,000.000  toyota   
6145   53865601                       Suspension -(Rear)  2,000.000  toyota   
6139   53853342  Wheel bearing spindle shaft-(Left rear)  2,000.000  toyota   
...         ...                                      ...        ...     ...   
7180   59230184                       Suspension -(Rear) 10,000.000  toyota   
5336   59245086                     Airbag krocksensor - 10,000.000  toyota   
5067   59230184                       Suspension -(Rear) 10,000.000  toyota   
6669   59230184                       Suspension -(Rear) 10,000.000  toyota   
5160   59240687              Trailing link rear -(Right) 10,000.000  toyota   

        model scrape_date  
6244  corolla  2026-02-

## Additional Formatting Cleanup

These final formatting adjustments standardize a small number of categorical text fields without changing the overall structure of the dataset.

In [101]:
# Standardize quality grade formatting
df["quality_grade"] = df["quality_grade"].astype("string").str.strip().str.upper()

# Check cleaned value counts
print(df["quality_grade"].value_counts(dropna=False))

quality_grade
A2    9716
A1    1300
A3     178
B2     123
B1      18
C       13
B3       7
C2       7
A        6
C1       6
Name: count, dtype: Int64


In [102]:
# Standardize subcategory text
df["subcategory"] = df["subcategory"].astype("string").str.strip().str.lower()

# Check cleaned value counts
print(df["subcategory"].value_counts(dropna=False).head(20))

subcategory
all                     1049
rear                     332
right                    297
left                     278
right rear               255
left rear                249
left front               202
right front              202
either side              138
caliper bracket          102
passenger airbag         101
gear box 5 speed         101
coil                     101
alternator               101
mass air-flow sensor     100
tank lid                 100
contact roll airbag      100
airbag control unit      100
fuel pump electric       100
automatic gear           100
Name: count, dtype: Int64


## Note on Subcategory

The `subcategory` field contains both true subtype labels and positional or location descriptors such as `left`, `rear`, and `right front`. It should therefore be interpreted as a mixed taxonomy field rather than a fully standardized part hierarchy.

## Save Cleaned Dataset

The file saved here is still an **initial cleaned version**. It reflects only conservative formatting and type fixes carried out in this notebook.

In [103]:
output_path = Path("/Users/riteshbhandari/Documents/Dokumentit – Ritesh - MacBook Pro/GitHub/DPPM/datasets/cleaned/clean_master_dataset.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(output_path, index=False)
print(f"Clean master dataset saved to: {output_path}")


Clean master dataset saved to: /Users/riteshbhandari/Documents/Dokumentit – Ritesh - MacBook Pro/GitHub/DPPM/datasets/cleaned/clean_master_dataset.csv
